# Ingest and link the machine-shop sample database

SQLite tables map to OWL classes; foreign keys map to object properties. Alias rows such as `立加01` merge into `VMC-01` by `assetCode`. `OperationExecution` is modeled as a temporal associative class.

In [ ]:
import sys
from pathlib import Path

HERE = Path.cwd()
if (HERE / "pipeline").exists():
    ROOT = HERE
else:
    ROOT = HERE / "cookbook" / "use_cases" / "manufacturing"
sys.path.insert(0, str(ROOT.parent))

from manufacturing.pipeline.ingest_and_link import (
    entity_by_attribute,
    ingest_and_link,
    neighbors,
)
from manufacturing.sample.build_sample_db import build_sample_db

In [ ]:
db_path = ROOT / "sample" / "machine_shop.sqlite"
build_sample_db(db_path, include_violations=False)
graph = ingest_and_link(db_path, include_violations=False, build_if_missing=False)
print(f"entities={len(graph['entities'])} relationships={len(graph['relationships'])}")
print(graph["metadata"]["associative_classes"][0])

In [ ]:
work_order = entity_by_attribute(graph, "workOrderNo", "WO-2026-001")
execution_id = neighbors(graph, work_order["id"], "belongsToWorkOrder", direction="in")[0]
print("work order", work_order["id"])
print("machine", neighbors(graph, execution_id, "executedOn"))
print("tool", neighbors(graph, execution_id, "usedTool"))
print("operator", neighbors(graph, execution_id, "performedBy"))
vmc = entity_by_attribute(graph, "assetCode", "VMC-01")
print("VMC-01 aliases", vmc.get("aliases"))